# Caliber — the decision workflow

You changed a prompt. Eval scores moved. Should you ship?

This notebook walks the full decision workflow on a simulated LLM eval:

1. `caliber.compare` — verdict + CI on paired scores
2. The four verdicts in action
3. `caliber.sample_size` — plan before you run
4. `caliber.benjamini_hochberg` — many metrics at once
5. `caliber.SequentialTester` — peek across batches

In [ ]:
import numpy as np
import caliber

print('caliber', caliber.__version__)

## 1. The basic verdict

Two prompt versions, scored by the same judge on the same 100 examples.
The new prompt is genuinely a little better (mean 0.65 vs 0.60).

In [ ]:
rng = np.random.default_rng(0)
old = rng.normal(0.60, 0.05, 100)
new = rng.normal(0.65, 0.05, 100)

result = caliber.compare(old, new, metric_name='accuracy')
print(f'verdict:           {result.verdict}')
print(f'mean difference:   {result.mean_difference:+.4f}')
print(f'95% CI:            {result.ci}')
print(f'p-value:           {result.p_value:.4g}')
print(f'method:            {result.method}')
print()
print(result.recommendation)

## 2. The four verdicts

Simulate one scenario for each verdict the library can return.

In [ ]:
rng = np.random.default_rng(1)

# BETTER — clear improvement.
old = rng.normal(0.60, 0.05, 100); new = rng.normal(0.75, 0.05, 100)
print('BETTER       ->', caliber.compare(old, new).verdict)

# WORSE — clear regression.
old = rng.normal(0.75, 0.05, 100); new = rng.normal(0.60, 0.05, 100)
print('WORSE        ->', caliber.compare(old, new).verdict)

# INCONCLUSIVE — too noisy at this n to call.
old = rng.normal(0.70, 0.20, 20); new = rng.normal(0.72, 0.20, 20)
print('INCONCLUSIVE ->', caliber.compare(old, new, seed=0).verdict)

# NO_CHANGE — real effect but below the practical threshold.
old = rng.normal(0.700, 0.005, 200); new = rng.normal(0.703, 0.005, 200)
print('NO_CHANGE    ->', caliber.compare(old, new, practical_threshold=0.05).verdict)

## 3. Auto-selection between paired-t and bootstrap

With `method='auto'` (the default), Caliber picks paired-t when n ≥ 30 and the
differences look roughly normal; otherwise it falls back to the bootstrap. You
don't have to think about it.

In [ ]:
small = caliber.compare(
    rng.normal(0.5, 0.1, 20),
    rng.normal(0.55, 0.1, 20),
    seed=0,
)
print(f'n=20  -> method={small.method}')

large = caliber.compare(
    rng.normal(0.5, 0.1, 100),
    rng.normal(0.55, 0.1, 100),
)
print(f'n=100 -> method={large.method}')

## 4. Plan before you run: `sample_size`

You can't decide *after* the eval whether your sample was big enough — that's
p-hacking. Decide *before*: what effect do you care about, and how confident
do you want to be?

In [ ]:
# I care about a 5-accuracy-point improvement, baseline judge noise σ=0.10,
# 80% power. How many paired evals do I need?
needed = caliber.sample_size(
    effect_size=0.05,
    effect_type='absolute',
    baseline_std=0.10,
    power=0.8,
)
print(f'n = {needed.n_per_arm}')

# Trade-offs — power vs sample size at the same effect:
for power in (0.7, 0.8, 0.9, 0.95):
    r = caliber.sample_size(
        effect_size=0.05, effect_type='absolute', baseline_std=0.10, power=power,
    )
    print(f'  power={power:.2f} -> n={r.n_per_arm}')

## 5. Many metrics at once: Benjamini-Hochberg

If you compare 20 metrics and call anything with p < 0.05 a win, you'll see
false positives. BH controls the false discovery rate.

In [ ]:
# Simulate 20 metrics: only 4 actually moved; the other 16 are noise.
rng = np.random.default_rng(7)
n = 50
true_winners = {0, 5, 10, 15}
p_values = []
for i in range(20):
    delta = 0.05 if i in true_winners else 0.0
    old = rng.normal(0.7, 0.1, n)
    new = rng.normal(0.7 + delta, 0.1, n)
    p_values.append(caliber.compare(old, new).p_value)

rejected = caliber.benjamini_hochberg(p_values, alpha=0.05)
naive    = [p < 0.05 for p in p_values]
print(f'naive p<0.05 rejected: {sum(naive)} (incl. false positives)')
print(f'BH-corrected:         {sum(rejected)} (FDR controlled)')
print('true winners caught by BH:',
      [i for i in true_winners if rejected[i]])

## 6. Peeking safely: `SequentialTester`

You're running evals batch by batch. You want to stop as soon as the answer
is clear — but the naïve "re-test at every batch" pattern inflates the
false-positive rate. Group-sequential testing fixes this with a boundary
that tightens at each look.

In [ ]:
tester = caliber.SequentialTester(max_n=500, n_looks=5, alpha=0.05)

rng = np.random.default_rng(0)
for k in range(5):
    batch_old = rng.normal(0.50, 0.10, 50)
    batch_new = rng.normal(0.55, 0.10, 50)
    result = tester.update(batch_old.tolist(), batch_new.tolist())
    print(f'look {k+1}/5  n={result.n:3d}  verdict={result.verdict}')
    if tester.is_done():
        print()
        print(result.recommendation)
        break

## Summary

Every workflow above is a single call. Caliber is intentionally a small
library: it solves one problem (is this change real?) with statistically
rigorous defaults, and gets out of the way.